# RavenStack SaaS Data Quality Assessment

## Purpose

This notebook performs pre-import data quality validation on the RavenStack SaaS dataset. Python is used only for data quality checks before loading the data into SQL. All final business analysis and metric calculations will be performed in SQL.

In [25]:
# Using pandas to read and inspect the CSV files 
# "pd" allows me to write pd.read_csv() instead of pandas.read_csv() 
import pandas as pd 
from pathlib import Path 

# Locate project root
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "python":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

accounts = pd.read_csv(DATA_DIR / "ravenstack_accounts.csv")
subscriptions = pd.read_csv(DATA_DIR / "ravenstack_subscriptions.csv")
feature_usage = pd.read_csv(DATA_DIR / "ravenstack_feature_usage.csv")
support_tickets = pd.read_csv(DATA_DIR / "ravenstack_support_tickets.csv")
churn_events = pd.read_csv(DATA_DIR / "ravenstack_churn_events.csv")

tables = {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "feature_usage": feature_usage,
    "support_tickets": support_tickets,
    "churn_events": churn_events
}

print("Files loaded successfully.")


Files loaded successfully.


In [26]:
# Dimension-Check
for table_name, df in tables.items():
    print(
        f"{table_name}: "
        f"{df.shape[0]:,} rows x {df.shape[1]} columns"
    )

accounts: 500 rows x 10 columns
subscriptions: 5,000 rows x 14 columns
feature_usage: 25,000 rows x 8 columns
support_tickets: 2,000 rows x 9 columns
churn_events: 600 rows x 9 columns


### Dimension Check Result

All five source CSV files loaded successfully. Their observed row and column counts match the documented RavenStack dataset dimensions. No obvious missing-row or missing-column issue was identified relative to the source documentation. 


## Column and Data Type Inspection

Inspect the column names and pandas-inferred data types for each source table. This check confirms that the loaded CSV structure match the documented source schemas before SQL import. 

In [27]:
# Check column and data types
for table_name, df in tables.items():
    print(f"\n--- {table_name.upper()} ---")
    print(df.dtypes)


--- ACCOUNTS ---
account_id           str
account_name         str
industry             str
country              str
signup_date          str
referral_source      str
plan_tier            str
seats              int64
is_trial            bool
churn_flag          bool
dtype: object

--- SUBSCRIPTIONS ---
subscription_id        str
account_id             str
start_date             str
end_date               str
plan_tier              str
seats                int64
mrr_amount           int64
arr_amount           int64
is_trial              bool
upgrade_flag          bool
downgrade_flag        bool
churn_flag            bool
billing_frequency      str
auto_renew_flag       bool
dtype: object

--- FEATURE_USAGE ---
usage_id                 str
subscription_id          str
usage_date               str
feature_name             str
usage_count            int64
usage_duration_secs    int64
error_count            int64
is_beta_feature         bool
dtype: object

--- SUPPORT_TICKETS ---
ticket_id

### Column and Data Type Check Result

All five tables contain the expected columns documented in the source README. The raw CSV files loaded with reasonable pandas-inferred data types.

**accounts**

- 10 columns 
- ID, text, categorical, and date fields loaded as `str`
- `seats` loaded as `int64`
- `is_trial` and `churn_flag` loaded as `bool`

**subscriptions**

- 14 columns
- ID, categorical, and date fields loaded as `str`
- `seats`, `mrr_amount`, and `arr_amount` loaded as `int64`
- trial, upgrade, downgrade, churn, and auto-renew flags loaded as `bool` 

**feature_usage**

- 8 columns
- IDs, usage date, and feature name loaded as `str`
- `usage_count`, `usage_duration_secs`, and `error_count` loaded as int64
- `is_beta_feature` loaded as `bool`

**support_tickets**

- 9 columns
- IDs, datetime fields, and priority loaded as `str`
- `resolution_time_hours` loaded as `float64`
- `first_response_time_minutes` loaded as `int64`
- `satisfaction_score` loaded as `float64`
- `escalation_flag` loaded as `bool`

**churn_events**

- 9 columns
- IDs, churn date, reason code, and feedback text loaded as `str`
- `refund_amount_usd` loaded as `float64`
- preceding upgrade, preceding downgrade, and reactivation flags loaded as `bool`

Text and date-like fields being loaded as `str` is expected when reading raw CSV files without explicit date conversion. These are pandas-inferred types only; Final SQLite data types have not yet been assigned.

In [28]:
for table_name, df in tables.items():
    print(f"\n{table_name}")
    print(df.columns.tolist())


accounts
['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag']

subscriptions
['subscription_id', 'account_id', 'start_date', 'end_date', 'plan_tier', 'seats', 'mrr_amount', 'arr_amount', 'is_trial', 'upgrade_flag', 'downgrade_flag', 'churn_flag', 'billing_frequency', 'auto_renew_flag']

feature_usage
['usage_id', 'subscription_id', 'usage_date', 'feature_name', 'usage_count', 'usage_duration_secs', 'error_count', 'is_beta_feature']

support_tickets
['ticket_id', 'account_id', 'submitted_at', 'closed_at', 'resolution_time_hours', 'priority', 'first_response_time_minutes', 'satisfaction_score', 'escalation_flag']

churn_events
['churn_event_id', 'account_id', 'churn_date', 'reason_code', 'refund_amount_usd', 'preceding_upgrade_flag', 'preceding_downgrade_flag', 'is_reactivation', 'feedback_text']


### All five tables contain the expected documented column

## NULL Value Assessment

Check each table for missing values and summarize only the columns that contain NULLs. Missing values are evaluated in context and are not automatically filled or removed.

In [29]:
#Run basic NULL checks
for table_name, df in tables.items():
    print(f"\n--- {table_name.upper()} ---")
    print(df.isna().sum())


--- ACCOUNTS ---
account_id         0
account_name       0
industry           0
country            0
signup_date        0
referral_source    0
plan_tier          0
seats              0
is_trial           0
churn_flag         0
dtype: int64

--- SUBSCRIPTIONS ---
subscription_id         0
account_id              0
start_date              0
end_date             4514
plan_tier               0
seats                   0
mrr_amount              0
arr_amount              0
is_trial                0
upgrade_flag            0
downgrade_flag          0
churn_flag              0
billing_frequency       0
auto_renew_flag         0
dtype: int64

--- FEATURE_USAGE ---
usage_id               0
subscription_id        0
usage_date             0
feature_name           0
usage_count            0
usage_duration_secs    0
error_count            0
is_beta_feature        0
dtype: int64

--- SUPPORT_TICKETS ---
ticket_id                        0
account_id                       0
submitted_at                

In [30]:
#Create a cleaner portfolio summary
null_summary = []

for table_name, df in tables.items():
    for column in df.columns:
        null_count = df[column].isna().sum()
        null_pct = (null_count / len(df)) * 100

        if null_count > 0:
            null_summary.append({
                "table": table_name,
                "column": column,
                "null_count": null_count,
                "null_percent": round(null_pct, 2)
            })

null_summary = pd.DataFrame(null_summary)

null_summary

,table,column,null_count,null_percent
0,subscriptions,end_date,4514,90.28
1,support_tickets,satisfaction_score,825,41.25
2,churn_events,feedback_text,148,24.67


### NULL Value Check Result

Missing values are limited to three columns in the current source files:

- `subscriptions.end_date`: 4,514 NULLs (90.28%)
- `support_tickets.satisfaction_score`: 825 NULLs (41.25%)
- `churn_events.feedback_text`: 148 NULLs (24.67%)

All other columns contain 0 NULL values.

The observed missing values are consistent with the source documentation:

- `end_date` is nullable for active subscriptions.
- `satisfaction_score` may be NULL when the customer did not provide a response.
- `feedback_text` is an optional customer comment.

These NULL values are therefore retained as meaningful missing values rather than automatically filled or removed.


## Candidate Key Uniqueness Assessment

Check whether the documented or intended identifier for each table is unique in the actual source data. Candidate IDs are evaluated before database constraints are assigned.

In [31]:
# Create candidate-key dictionary
primary_key_candidates = {
    "accounts": "account_id",
    "subscriptions": "subscription_id",
    "feature_usage": "usage_id",
    "support_tickets": "ticket_id",
    "churn_events": "churn_event_id"
}

duplicate_results = []

for table_name, key in primary_key_candidates.items():
    df = tables[table_name]
    duplicate_results.append({
        "table": table_name,
        "candidate_key": key,
        "rows": len(df),
        "unique_ids": df[key].nunique(),
        "duplicate_rows": df.duplicated(subset=[key]).sum()
    })

duplicate_results = pd.DataFrame(duplicate_results)

duplicate_results


,table,candidate_key,rows,unique_ids,duplicate_rows
0,accounts,account_id,500,500,0
1,subscriptions,subscription_id,5000,5000,0
2,feature_usage,usage_id,25000,24979,21
3,support_tickets,ticket_id,2000,2000,0
4,churn_events,churn_event_id,600,600,0


In [ ]:
# Inspecting every affected row 
duplicate_usage_ids = feature_usage[
    feature_usage.duplicated(
        subset=["usage_id"],
        keep=False
    )
].sort_values("usage_id")

duplicate_usage_ids

# Resulting in 42 rows (because of 21 duplicated IDs, each occurs 2 times) 

,usage_id,subscription_id,usage_date,feature_name,usage_count,usage_duration_secs,error_count,is_beta_feature
19294,U-0c9318,S-01b2dc,2023-12-11,feature_9,5,1060,3,False
17533,U-0c9318,S-0ffab0,2024-01-30,feature_11,3,1614,0,False
20588,U-13ce5b,S-9b623b,2023-03-26,feature_28,10,2050,1,False
9626,U-13ce5b,S-8b0950,2024-09-10,feature_9,8,824,0,True
18480,U-2103bb,S-7fc49b,2023-04-18,feature_12,9,5022,1,False
10379,U-2103bb,S-ae3270,2024-01-06,feature_3,10,5510,0,False
22,U-25b56c,S-810c27,2024-10-06,feature_20,7,231,0,False
7574,U-25b56c,S-34253c,2023-10-28,feature_20,6,2166,1,False
21376,U-48a4aa,S-93f835,2024-08-24,feature_39,10,4770,0,False
1085,U-48a4aa,S-383ac2,2023-02-12,feature_28,14,7126,0,False


In [33]:
# Extra check for duplicate rows
duplicate_usage_ids.duplicated().sum()

np.int64(0)

### Export Duplicate Usage ID QA Evidence

The duplicated `usage_id` records identified during QA are exported for
documentation and review. The source data are not modified, deleted, or
corrected during this Python QA stage.

In [ ]:
# Save duplicate usage ID evidence for QA documentation
QA_DIR = PROJECT_ROOT / "output" / "qa"
QA_DIR.mkdir(parents=True, exist_ok=True)

duplicate_usage_ids.to_csv(
    QA_DIR / "duplicate_usage_ids.csv",
    index=False
)

print(f"Rows exported: {len(duplicate_usage_ids)}")
print(f"Saved to: {QA_DIR / 'duplicate_usage_ids.csv'}")

### Candidate Key Uniqueness Check Result

The candidate identifier columns for `accounts`, `subscriptions`, `support_tickets`, and `churn_events` are unique in the current source files.

A data-quality issue was identified in `feature_usage`:

- `feature_usage` contains 25,000 rows.
- `usage_id` contains 24,979 unique values.
- 21 `usage_id` values are duplicated.
- Each duplicated `usage_id` occurs twice, affecting 42 total records.
- The affected records are not exact duplicate rows; distinct usage-event records share the same identifier.

The source README describes `usage_id` as a unique usage-event identifier, but this uniqueness condition does not hold in the actual CSV.

No duplicate records were deleted or renumbered. The raw source data is preserved unchanged, and the treatment of `usage_id` will be addressed during database schema design.


## Foreign-Key Integrity Assessment

Validate that all documented foreign-key values in the child tables reference existing records in their corresponding parent tables. Any unmatched child records would be considered orphan records.

In [34]:
# Foreign Key Checks
fk_checks = {
    "subscriptions.account_id -> accounts.account_id":
        (~subscriptions["account_id"]
         .isin(accounts["account_id"])).sum(),

    "feature_usage.subscription_id -> subscriptions.subscription_id":
        (~feature_usage["subscription_id"]
         .isin(subscriptions["subscription_id"])).sum(),

    "support_tickets.account_id -> accounts.account_id":
        (~support_tickets["account_id"]
         .isin(accounts["account_id"])).sum(),

    "churn_events.account_id -> accounts.account_id":
        (~churn_events["account_id"]
         .isin(accounts["account_id"])).sum()
}

# Print orphan record counts for each foreign key relationship
# orphan records = child record whose foreign-key value points to a non-existent parent record
for relationship, orphan_count in fk_checks.items():
    print(f"{relationship}: {orphan_count} orphan records")

subscriptions.account_id -> accounts.account_id: 0 orphan records
feature_usage.subscription_id -> subscriptions.subscription_id: 0 orphan records
support_tickets.account_id -> accounts.account_id: 0 orphan records
churn_events.account_id -> accounts.account_id: 0 orphan records


### Foreign-Key Integrity Check Result

All four documented foreign-key relationships passed referential-integrity validation with zero orphan records:

- `subscriptions.account_id` -> `accounts.account_id`: 0 orphan records
- `feature_usage.subscription_id` -> `subscriptions.subscription_id`: 0 orphan records
- `support_tickets.account_id` -> `accounts.account_id`: 0 orphan records
- `churn_events.account_id` -> `accounts.account_id`: 0 orphan records

This confirms that all child records reference existing parent records for the documented relationships in the current source files.

The foreign-key validation result is independent of the duplicate `usage_id` issue identified earlier in `feature_usage`; the subscription references remain valid even though `usage_id` is not unique.


## Value and Temporal Logic Assessment

Validate date parsing, categorical values, numeric ranges, cross-field relationships, and cross-table temporal consistency. These checks identify values that may be structurally valid but inconsistent with expected business logic.

In [35]:
# Convert date columns to pandas datetime
accounts["signup_date"] = pd.to_datetime(
    accounts["signup_date"],
    errors="coerce"
)

subscriptions["start_date"] = pd.to_datetime(
    subscriptions["start_date"],
    errors="coerce"
)

subscriptions["end_date"] = pd.to_datetime(
    subscriptions["end_date"],
    errors="coerce"
)

feature_usage["usage_date"] = pd.to_datetime(
    feature_usage["usage_date"],
    errors="coerce"
)

support_tickets["submitted_at"] = pd.to_datetime(
    support_tickets["submitted_at"],
    errors="coerce"
)

support_tickets["closed_at"] = pd.to_datetime(
    support_tickets["closed_at"],
    errors="coerce"
)

churn_events["churn_date"] = pd.to_datetime(
    churn_events["churn_date"],
    errors="coerce"
)

In [36]:
# Date Null Checks
date_null_check = {
    "accounts.signup_date":
        accounts["signup_date"].isna().sum(),

    "subscriptions.start_date":
        subscriptions["start_date"].isna().sum(),

    "subscriptions.end_date":
        subscriptions["end_date"].isna().sum(),

    "feature_usage.usage_date":
        feature_usage["usage_date"].isna().sum(),

    "support_tickets.submitted_at":
        support_tickets["submitted_at"].isna().sum(),

    "support_tickets.closed_at":
        support_tickets["closed_at"].isna().sum(),

    "churn_events.churn_date":
        churn_events["churn_date"].isna().sum()
}

date_null_check

# Categorical Null Checks
print("Industries:")
print(sorted(accounts["industry"].unique()))

print("\nReferral sources:")
print(sorted(accounts["referral_source"].unique()))

print("\nAccount plan tiers:")
print(sorted(accounts["plan_tier"].unique()))

print("\nSubscription plan tiers:")
print(sorted(subscriptions["plan_tier"].unique()))

print("\nBilling frequencies:")
print(sorted(subscriptions["billing_frequency"].unique()))

print("\nTicket priorities:")
print(sorted(support_tickets["priority"].unique()))

print("\nChurn reasons:")
print(sorted(churn_events["reason_code"].unique()))

Industries:
['Cybersecurity', 'DevTools', 'EdTech', 'FinTech', 'HealthTech']

Referral sources:
['ads', 'event', 'organic', 'other', 'partner']

Account plan tiers:
['Basic', 'Enterprise', 'Pro']

Subscription plan tiers:
['Basic', 'Enterprise', 'Pro']

Billing frequencies:
['annual', 'monthly']

Ticket priorities:
['high', 'low', 'medium', 'urgent']

Churn reasons:
['budget', 'competitor', 'features', 'pricing', 'support', 'unknown']


In [37]:
# Numeric Null Checks
numeric_checks = {
    "accounts seats <= 0":
        (accounts["seats"] <= 0).sum(),

    "subscriptions seats <= 0":
        (subscriptions["seats"] <= 0).sum(),

    "negative MRR":
        (subscriptions["mrr_amount"] < 0).sum(),

    "negative ARR":
        (subscriptions["arr_amount"] < 0).sum(),

    "negative usage count":
        (feature_usage["usage_count"] < 0).sum(),

    "negative usage duration":
        (feature_usage["usage_duration_secs"] < 0).sum(),

    "negative error count":
        (feature_usage["error_count"] < 0).sum(),

    "negative resolution time":
        (support_tickets["resolution_time_hours"] < 0).sum(),

    "negative first response time":
        (support_tickets["first_response_time_minutes"] < 0).sum(),

    "negative refund":
        (churn_events["refund_amount_usd"] < 0).sum()
}

for check, count in numeric_checks.items():
    print(f"{check}: {count}")

accounts seats <= 0: 0
subscriptions seats <= 0: 0
negative MRR: 0
negative ARR: 0
negative usage count: 0
negative usage duration: 0
negative error count: 0
negative resolution time: 0
negative first response time: 0
negative refund: 0


In [38]:
# Data Consistency Checks
(
    subscriptions["arr_amount"]
    ==
    subscriptions["mrr_amount"] * 12
).all()

np.True_

In [39]:
# Basic data relationships check
subscription_dates = subscriptions.merge(
    accounts[["account_id", "signup_date"]],
    on="account_id",
    how="left"
)

(
    subscription_dates["start_date"]
    <
    subscription_dates["signup_date"]
).sum()

np.int64(0)

In [40]:
# End Date Consistency Check
(
    subscriptions["end_date"].notna()
    &
    (
        subscriptions["end_date"]
        <
        subscriptions["start_date"]
    )
).sum()

np.int64(0)

In [41]:
# Support Ticket Date Consistency Check
(
    support_tickets["closed_at"]
    <
    support_tickets["submitted_at"]
).sum()

np.int64(0)

In [42]:
# Churn Date Consistency Check
churn_dates = churn_events.merge(
    accounts[["account_id", "signup_date"]],
    on="account_id",
    how="left"
)

(
    churn_dates["churn_date"]
    <
    churn_dates["signup_date"]
).sum()

np.int64(0)

In [43]:
# Feature Usage Date Consistency Check
usage_dates = feature_usage.merge(
    subscriptions[
        ["subscription_id", "start_date", "end_date"]
    ],
    on="subscription_id",
    how="left"
)

In [44]:
# Check usage before subscription start
(
    usage_dates["usage_date"]
    <
    usage_dates["start_date"]
).sum()

np.int64(19142)

In [45]:
# Check usage after subscription end
(
    usage_dates["end_date"].notna()
    &
    (
        usage_dates["usage_date"]
        >
        usage_dates["end_date"]
    )
).sum()

np.int64(290)

In [46]:
# Check tickets before account signup
ticket_dates = support_tickets.merge(
    accounts[["account_id", "signup_date"]],
    on="account_id",
    how="left"
)

(
    ticket_dates["submitted_at"]
    <
    ticket_dates["signup_date"]
).sum()

np.int64(1077)

In [47]:
# Count churn events per account
churn_event_counts = (
    churn_events
    .groupby("account_id")
    .size()
    .rename("churn_event_count")
    .reset_index()
)

# Merge counts into accounts
account_churn_check = accounts[
    ["account_id", "churn_flag"]
].merge(
    churn_event_counts,
    on="account_id",
    how="left"
)

# Replace missing derived counts with 0
account_churn_check["churn_event_count"] = (
    account_churn_check["churn_event_count"]
    .fillna(0)
)

# Identify False flags with churn events
false_with_event = (
    (~account_churn_check["churn_flag"])
    &
    (account_churn_check["churn_event_count"] > 0)
).sum()

# Identify True flags without churn events
true_without_event = (
    (account_churn_check["churn_flag"])
    &
    (account_churn_check["churn_event_count"] == 0)
).sum()

# Print both results
print(
    "churn_flag=False but churn event exists:",
    false_with_event
)

print(
    "churn_flag=True but no churn event exists:",
    true_without_event
)

churn_flag=False but churn event exists: 277
churn_flag=True but no churn event exists: 35


### Value and Temporal Logic Check Result

Date fields were successfully parsed for validation without introducing unexpected missing values. The existing NULL values in `subscriptions.end_date` were preserved.

Categorical-value inspection identified no obvious malformed or unexpected values. Numeric range checks also passed: no non-positive seat counts or negative revenue, usage, duration, response-time, error-count, or refund values were identified. In addition, `arr_amount` equals `mrr_amount × 12` for all subscription records.

Basic date-order checks passed:

- No subscriptions begin before their associated account signup date.
- No populated subscription end dates occur before subscription start dates.
- No support tickets close before their submission timestamp.
- No churn events occur before the associated account signup date.

However, several cross-table temporal and semantic inconsistencies were identified:

- 19,142 feature-usage records occur before the referenced subscription start date.
- 290 feature-usage records occur after the referenced subscription end date.
- 1,077 support tickets were submitted before the associated account signup date.
- 277 accounts have `churn_flag=False` despite having at least one churn-event record.
- 35 accounts have `churn_flag=True` but no corresponding churn-event record.

These issues do not represent foreign-key failures; the referenced parent records exist. Instead, they represent temporal or cross-table semantic inconsistencies in the source data.

No source records were deleted, corrected, or overwritten as part of this assessment.


## Data Quality Findings

### Usage ID Uniqueness Issue

The source documentation describes `usage_id` in the `feature_usage` table as a unique identifier for each usage event. However, the Python data quality assessment identified 25,000 feature-usage records but only 24,979 unique `usage_id` values. A total of 21 `usage_id` values occur twice, producing 42 records associated with duplicated identifiers.

Inspection of these records indicates that they are not simply exact duplicate rows; the repeated identifiers can represent different usage-event records. Therefore, no records were deleted, renumbered, or otherwise modified during the Python quality-assurance stage. The original raw dataset has been preserved.

This issue will be considered during database modeling before assigning the primary key for the `feature_usage` table. Python is used only to identify and document the issue; the raw source data remains unchanged.
